# 3D multichannel segmentation

This notebook demonstrates a complete workflow for segmenting nuclei, cytoplasm, and intracellular structures from **3D microscopy data**, and for quantifying structures at both the object and cell level. The pipeline is modular and extensible, allowing different segmentation strategies depending on the channel and biological question.

The general pipeline includes:

1. Loading and preprocessing volumetric data

    - Intensity normalization
    - Optional downsampling and smoothing
2. Segmentation of cellular compartments

    - Nuclei (using deep learning models: StarDist or Cellpose)

    - Cytoplasm (via intensity- or membrane-based watershed)

    - Detection of intracellular structures

        -AICS segmentation workflows (e.g., dot/filament filters)
3. (optional) Post-processing (object removal, smoothing)

4. Quantification of structures per cell

    - Mapping detected objects to their parent cell

    - Extracting per-object and per-cell features (count, volume, intensity)

    - Exporting results to CSV for downstream analysis

5. Quality control

    - Visual overlays (e.g., maximum intensity projections, per-cell structure maps)
    
The goal is to link each detected intracellular structure to its cell of origin, enabling biologically meaningful, cell-level measurements.

## Load a 3D image 

You need to specify your input folder, how many images to segment, if you want to dowsample, normalise per slice, and the map your channel into a directory:

    - "nucleus": 0 → channel index 0 contains the nuclear stain.

    - "cytoplasm": 2 → channel index 2 contains cytoplasmic signal.

    - "intracellular": 1 → channel index 1 contains the organelle/structure of interest.
    
This mapping depends on the acquisition setup and might change between datasets — check the raw image metadata if in doubt (e.g. via opening a representative example image in ImageJ)

In [ ]:
## Libraries
## Load packages
# import glob, os
import skimage
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pandas as pd
import os
import tifffile

# --------------------
# INPUT SETTINGS
# --------------------

input_folder: "/home/ucbtvsi/Image-Analysis-Summer-Project/raw/images"      # Directory containing TIFF image frames
output_folder: "/home/ucbtvsi/Image-Analysis-Summer-Project/output"         # Directory where all outputs will be saved

# --------------------
# --------------------

# 1. load images as multichannel dictionary

# add a dictionary for your images, stating which channel corresponds to which of the structures
channel_map = {
    "nucleus": 0,
    "cytoplasm": 2,
    "intracellular": 1
}

# enter voxel size of your images (obtained from the metadata)
voxel_size_um = [1, 0.48, 0.48 ] # Z, Y, X; from metadata

# loading images from the input folder directory
files = sorted(input_folder.glob("*.tif"))

if not files:
    raise FileNotFoundError("No .tif files found in the folder.")

all_volumes = []

for f in files:
    img = tifffile.imread(f)  # could be (Z,Y,X,C) or (C,Z,Y,X)

    # this is to handle errors
    if img.ndim != 4:
        raise ValueError(f"Unexpected image shape: {img.shape}. Expected 4D (Z,Y,X,C) or (C,Z,Y,X).")
    
    # identify channels axis and move it to last
    if img.shape[-1] <= 10:
            zyx_channels = img  # already in (Z,Y,X,C)
    else:
        # guess channel axis (the one with size < 10)
        channel_axis = np.argmin(img.shape)
        if img.shape[channel_axis] < 10:
                zyx_channels = np.moveaxis(img, channel_axis, -1) # moving the channel axis to the last position
        else:
            raise ValueError(f"Cannot determine channel axis for shape {img.shape}")

    # map channels to names
    channels_dict = {}
    for name, idx in channel_map.items():
        if idx >= zyx_channels.shape[-1]:
            raise IndexError(f"Channel index {idx} for '{name}' out of range in image {f.name}")
        channels_dict[name] = zyx_channels[..., idx]

    # append structured entry with filename and channels (per image)
    all_volumes.append({
        "filename": f.stem, 
        "channels": channels_dict
    })

# preview loaded images: first image [0] as example
print(all_volumes[0]["filename"])       # e.g., "sample01.tif"
print(all_volumes[0]["channels"].keys()) # what channels are included in the dictionary? should be: dict_keys(['nucleus', 'cytoplasm', 'intracellular'])
print(all_volumes[0]["channels"]["nucleus"].shape)  # (Z, Y, X)

## Quick check: middle slice for each channel

We can display the **middle Z-slice** of each channel from the same (first) image of the folder. You can view different images by editing Volume_number.

This allows you to quickly confirm:

    - Channel order (e.g., nucleus, cytoplasm, intracellular)
    - The objects in the different channels should overlap, with nucleus and intracellular objects within the cytoplasm area.
    - That the data is loaded correctly and matches expectations
    
If the signal looks wrong (e.g., nucleus is empty but cytoplasm is bright), check the channel_map definition above.

In [ ]:
### Which volume to view?
Volume_number = 0

# take the first loaded volume
vol = all_volumes[Volume_number]["channels"]

# find middle slice
z_mid = next(iter(vol.values())).shape[0] // 2  

# plot all channels in a grid
n_channels = len(vol)
fig, axes = plt.subplots(1, n_channels, figsize=(5 * n_channels, 5))

for ax, (name, data) in zip(axes, vol.items()):
    ax.imshow(data[z_mid], cmap="gray")
    ax.set_title(f"{name} (z={z_mid})")
    ax.axis("off")

plt.tight_layout()
plt.show()

### Per-channel preprocessing, segmentation, and quantification

 Each channel is preprocessed and segmented in a separate sub-section, allowing to save and visualise intermediate steps for quality checks.

We need to define:

1. Number of images to test (num_test_volumes): Defines how many test images from the input_folder will be used for the trial analysis throughout
2. Downsampling (downsize_factor): Reduces memory usage and speeds up computation.
3. Preprocessing: Gaussian filtering and normalization (either per-slice or across the full 3D stack).

In [ ]:
# --------------------
# INPUT SETTINGS
# --------------------

### How many images to segment? More takes longer but gives more info
num_test_volumes = 1

### downsieze factor
downsize_factor = 1         #scaling factor or 1 to keep original                            
per_slice_norm = True       # True = normalize per-slice, False = normalize whole stack